## Integration check — OOF probabilities → expected value → budget allocation

End-to-end validation of the S2 → S3 chain against a known structural
property: with **constant** parameters (same cost and same effectiveness for
every customer), `expected_value` is just `p * m` times a positive scalar, and
the budget buys a fixed number of slots. The knapsack therefore has no room to
be clever — its selection must coincide exactly with the top-k of the `p * m`
ranking.

Any mismatch means a bug in the chain, not a smarter allocation.

In [1]:
import numpy as np
import pandas as pd

from retention_optimizer.optimization.allocator import allocate
from retention_optimizer.optimization.value import expected_value

BUDGET = 10_000.0
COST = 20.0  # constant cost per retention action
TOP_K = int(BUDGET // COST)  # slots the budget buys = 500

oof = pd.read_csv('../data/processed/oof_predictions.csv')

p = oof['p_oof'].to_numpy()             # calibrated OOF P(churn)
m = oof['MonthlyCharges'].to_numpy()    # monthly margin proxy

print(f'customers: {len(p)}  |  budget: {BUDGET:,.0f} EUR  |  slots: {TOP_K}')
oof.head()

customers: 7043  |  budget: 10,000 EUR  |  slots: 500


,customerID,p_oof,MonthlyCharges
0,7590-VHVEG,0.738075,29.85
1,5575-GNVDE,0.044292,56.95
2,3668-QPYBK,0.394004,53.85
3,7795-CFOCW,0.062009,42.30
4,9237-HQITU,0.668407,70.70


### 1. Ranking baseline: the top-k by `priority_score = p * m`

In [2]:
priority_score = p * m

# Indices of the TOP_K highest scores (descending).
top_idx = np.argsort(priority_score)[::-1][:TOP_K]
top_set = set(top_idx.tolist())

print(f'top-{TOP_K} score range: '
      f'{priority_score[top_idx].min():.4f} .. {priority_score[top_idx].max():.4f}')
print(f'first excluded score:  {np.sort(priority_score)[::-1][TOP_K]:.4f}')

top-500 score range: 57.8337 .. 91.8481
first excluded score:  57.8186


### 2. Optimizer: what the knapsack actually picks

In [3]:
mask, info = allocate(p, m, budget=BUDGET, cost=COST)

selected_idx = np.flatnonzero(mask)
selected_set = set(selected_idx.tolist())

print(f'selected: {len(selected_idx)} customers')

selected: 500 customers


### 3. Central verification — the two sets must be identical

In [1]:
overlap = len(top_set & selected_set)

print(f'OVERLAP: {overlap}/{TOP_K}  ({overlap / TOP_K:.1%})')

if overlap == TOP_K:
    print('PASS — the knapsack selection is exactly the top-k of p * m. IF we do not implement variable cost per customer or variable uplift, this is expected')
else:
    missing = sorted(top_set - selected_set)   # ranked in, not selected
    extra = sorted(selected_set - top_set)     # selected, not ranked in
    print(f'FAIL — {len(missing)} ranked-in customers were not selected.')
    print('in top-k but NOT selected:', missing)
    print('selected but NOT in top-k:', extra)
    for i in missing[:10]:
        print(f'  idx {i}: p={p[i]:.4f} m={m[i]:.2f} score={priority_score[i]:.4f}')

NameError: name 'top_set' is not defined

### 4. Allocation summary

In [5]:
print(f"n_selected     : {info['n_selected']}")
print(f"spend          : {info['spend']:,.2f} EUR")
print(f"expected_value : {info['expected_value']:,.2f} EUR")
print()
print(f"n_selected == {TOP_K} : {info['n_selected'] == TOP_K}")
print(f"spend == budget    : {np.isclose(info['spend'], BUDGET)}")
print(f"ROI (value/spend)  : {info['expected_value'] / info['spend']:.2f}x")

n_selected     : 500
spend          : 10,000.00 EUR
expected_value : 113,385.01 EUR

n_selected == 500 : True
spend == budget    : True
ROI (value/spend)  : 11.34x


### 5. Horizon sweep — does the net-value filter ever bite?

A customer is only worth acting on if `expected_value > cost`. Shortening the
horizon `H` shrinks every expected value, so at some point the filter should
start rejecting customers and leave part of the budget unspent
(`n_selected < TOP_K`, `spend < BUDGET`).

The last column counts how many customers clear the filter at all: the budget
binds instead of the filter whenever that count exceeds the number of slots.

In [6]:
for H in (6, 12, 24):
    mask_h, info_h = allocate(p, m, budget=BUDGET, cost=COST, H=H)
    n_viable = int((expected_value(p, m, H=H) > COST).sum())
    print(
        f'H={H:2d} | n_selected={info_h["n_selected"]:3d} '
        f'| spend={info_h["spend"]:9,.2f} '
        f'| expected_value={info_h["expected_value"]:11,.2f} '
        f'| customers above cost: {n_viable}'
    )

H= 6 | n_selected=500 | spend=10,000.00 | expected_value=  58,384.33 | customers above cost: 3390


H=12 | n_selected=500 | spend=10,000.00 | expected_value= 113,385.01 | customers above cost: 4380


H=24 | n_selected=500 | spend=10,000.00 | expected_value= 214,008.44 | customers above cost: 5037
